# Llama-3-8B: Исследование структуры слоев

В этом блокноте мы разберем архитектуру Llama-3-8B по слоям, используя `transformers` и `torch`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoConfig
import matplotlib.pyplot as plt
import numpy as np

model_id = "meta-llama/Meta-Llama-3-8B"
print(f"Модель: {model_id}")

## 1. Загрузка конфигурации (Meta Device)

Мы загружаем модель на `meta` девайс, чтобы увидеть структуру без загрузки 15GB весов в RAM.

In [ ]:
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
with torch.device("meta"):
    model = AutoModelForCausalLM.from_config(config)

print(f"Количество слоев: {len(model.model.layers)}")
print(f"Размерность эмбеддинга: {config.hidden_size}")

## 2. Полная структура слоев

Посмотрим на типичный блок Llama-3.

In [ ]:
layer0 = model.model.layers[0]
print("Структура LlamaDecoderLayer:")
print(layer0)

## 3. Внутри Self-Attention

Инспектируем проекции Q, K, V и Output.

In [ ]:
attn = layer0.self_attn
print(f"Q projection: {attn.q_proj}")
print(f"K projection: {attn.k_proj}")
print(f"V projection: {attn.v_proj}")
print(f"O projection: {attn.o_proj}")

print(f"\nПараметры MLP:")
print(f"Gate projection: {layer0.mlp.gate_proj}")
print(f"Up projection:   {layer0.mlp.up_proj}")
print(f"Down projection: {layer0.mlp.down_proj}")

## 4. Эмуляция SVD для 4096x4096матрицы

Так как на `meta` девайсе нет реальных чисел, мы создадим случайную матрицу с таким же распределением, чтобы показать метод анализа.

In [ ]:
# Эмуляция весов q_proj (4096, 4096)
W_sim = torch.randn(4096, 4096) * 0.02

U, S, V = torch.svd(W_sim)
S_np = S.numpy()

plt.figure(figsize=(10, 4))
plt.semilogy(S_np)
plt.title("Логарифмический спектр сингулярных чисел (Llama-3-8B Simulated)")
plt.grid(True)
plt.show()

print("Если спектр падает экспоненциально, мы можем сжать слой без потери точности.")